# 03 - Global Pointer and BERT + Global Context

Notebook này chạy 2 hướng còn lại:

- Global Pointer (Tâm): dự đoán span entity.
- BERT + Global Context (Hoàng): BERT + BiLSTM context layer + token classifier.

In [ ]:
from pathlib import Path

CANDIDATES = [
    Path.cwd(),
    Path('/content/NLP-test'),
    Path('/content/drive/MyDrive/NLP-test'),
    Path('/Users/kodomotachi/specialist/NLP-test'),
]
ROOT = next((path for path in CANDIDATES if (path / 'src').exists()), None)
assert ROOT is not None, 'Cannot find project root. In Colab, upload/copy NLP-test to /content/NLP-test or /content/drive/MyDrive/NLP-test.'
%cd $ROOT

In [ ]:
import importlib
import src.deep_ner_models as deep_ner_models
importlib.reload(deep_ner_models)
from src.deep_ner_models import load_jsonl, build_label_map, train_global_pointer, train_custom_sequence_model

train_rows = load_jsonl('data/processed/b_unique.jsonl')
valid_rows = load_jsonl('data/processed/b_valid.jsonl')
test_rows = load_jsonl('data/processed/b_test.jsonl')
label2id, id2label = build_label_map(train_rows + valid_rows + test_rows)

In [ ]:
LIMIT_TRAIN = None
LIMIT_EVAL = None
EPOCHS = 3
MAX_LENGTH = 384
GLOBAL_POINTER_BATCH_SIZE = 2
GLOBAL_CONTEXT_BATCH_SIZE = 2

train_run = train_rows[:LIMIT_TRAIN] if LIMIT_TRAIN else train_rows
valid_run = valid_rows[:LIMIT_EVAL] if LIMIT_EVAL else valid_rows
test_run = test_rows[:LIMIT_EVAL] if LIMIT_EVAL else test_rows

## Global Pointer (Tâm)

In [ ]:
global_pointer_result = train_global_pointer(
    name='tam_bert_global_pointer',
    model_name='bert-base-uncased',
    train_rows=train_run,
    valid_rows=valid_run,
    test_rows=test_run,
    label2id=label2id,
    epochs=EPOCHS,
    batch_size=GLOBAL_POINTER_BATCH_SIZE,
    max_length=MAX_LENGTH,
    threshold=0.0,
)
global_pointer_result

## BERT + Global Context (Hoàng)

In [ ]:
global_context_result = train_custom_sequence_model(
    name='hoang_bert_global_context',
    architecture='global_context',
    model_name='bert-base-uncased',
    train_rows=train_run,
    valid_rows=valid_run,
    test_rows=test_run,
    label2id=label2id,
    id2label=id2label,
    epochs=EPOCHS,
    batch_size=GLOBAL_CONTEXT_BATCH_SIZE,
    max_length=MAX_LENGTH,
)
global_context_result